In [0]:
import requests
import pandas as pd 
import pyspark as ps
import time
from pyspark.sql.functions import col, sum as spark_sum, isnan, when, count, trim
from pyspark.sql import SparkSession
from pyspark.sql.types import NullType

In [0]:
indicators = ['NCDMORT3070', 'SDGPM25', 'WHOSIS_000001', 'WHOSIS_000002', 'MDG_0000000029', 'HIV_0000000026', 'NCD_GLUC_04', 'BP_04', 'NCD_BMI_30C', 'M_Est_smk_curr_std', 'WSH_WATER_SAFELY_MANAGED', 'WSH_SANITATION_SAFELY_MANAGED', 'NUTRITION_ANT_HAZ_NE2', 'NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV', 'GHED_GGHE-D_pc_US_SHA2011']

In [0]:
base_url = "https://ghoapi.azureedge.net/api/"
dfs = []
erros = []
viewed_cols = []

for code in indicators:
    url = f"{base_url}{code}"
    try:
        response = requests.get(url, timeout=30)
        data = response.json()
        if not data.get('value'):
            print(f"{code} — sem dados, pulando")
            erros.append(code)
            continue

        df_temp = pd.DataFrame(data['value'])
        for col in viewed_cols:
            if col not in df_temp.columns:
                df_temp[col] = None
        dfs.append(df_temp)
        print(f"{code} — {len(df_temp)} linhas")
        for col in df_temp.columns:
            if col not in viewed_cols:
                viewed_cols.append(col)
    except Exception as e:
        print(f"{code} — erro: {e}")
        erros.append(code)
    
    time.sleep(2)

df_pandas = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {df_pandas.shape}")
print(f"Erros: {erros}")

In [0]:
spark = SparkSession.builder.getOrCreate()

df_spark = spark.table("workspace.bronze.gho_raw").drop("DataSourceDimType", "DataSourceDim")

df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.gho_raw")

In [0]:
tables = spark.sql("SHOW TABLES IN workspace.bronze").collect()

In [0]:
for table in tables:
    name = table.tableName

    if name == "gho_raw":
        continue
    
    df = spark.table(f"workspace.bronze.{name}")
    print(f"{name} - {df.count()} lines")

In [0]:
spark.sql("SHOW CATALOGS").show()

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show()

In [0]:
spark.sql("SHOW TABLES IN workspace.bronze").show()
spark.sql("SHOW TABLES IN workspace.default").show()

In [0]:
spark.sql("DESCRIBE EXTENDED workspace.bronze.gho_raw").show(truncate=False)

In [0]:
spark.sql("DESCRIBE HISTORY workspace.bronze.gho_raw").show(5, truncate=False)

In [0]:
df = spark.table("workspace.bronze.gho_raw")

colunas_sem_problema = [c for c in df.columns 
                        if c not in ("DataSourceDimType", "DataSourceDim")]

df_safe = df.select(colunas_sem_problema)
pdf = df_safe.toPandas()

total = len(pdf)
null_counts = pdf.isnull().sum()

print(f"Total rows: {total}\n")
print(null_counts.to_string())